In [ ]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# ==================================================
# CONFIG
# ==================================================

POU_FILE = "/content/1739240888.csv"
FOOD_FILE = "/content/1759725293.csv"
NUTRITION_FILE = "/content/nutrition.csv"

HOST = "mysql-1e4ba5e3-data-engineering-1945.l.aivencloud.com"
PORT = 13140
DATABASE = "healty-analytic"
USER = "avnadmin"
PASSWORD = "AVNS_4dfoudlRmPHRBDs8CnH"


# ==================================================
# EXTRACT
# ==================================================

def extract_data():
    df_pou = pd.read_csv(POU_FILE)
    df_food = pd.read_csv(FOOD_FILE)
    df_nutrition = pd.read_csv(NUTRITION_FILE)

    print("=== DATA LOADED ===")
    print("POU       :", df_pou.shape)
    print("FOOD      :", df_food.shape)
    print("NUTRITION :", df_nutrition.shape)

    return df_pou, df_food, df_nutrition


# ==================================================
# CLEANING
# ==================================================

def clean_dataframe(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.drop_duplicates()

    df = df.loc[:, ~df.columns.str.contains("^unnamed")]

    return df


def transform_data(df_pou, df_food, df_nutrition):

    df_pou = clean_dataframe(df_pou)
    df_food = clean_dataframe(df_food)
    df_nutrition = clean_dataframe(df_nutrition)

    df_pou = df_pou.drop(
        columns=["no"],
        errors="ignore"
    )

    df_food = df_food.drop(
        columns=["no"],
        errors="ignore"
    )

    return df_pou, df_food, df_nutrition


# ==================================================
# FACT TABLE
# ==================================================

def build_fact_table(df_food, df_pou):

    fact_table = pd.merge(
        df_food,
        df_pou,
        on=["tahun", "kode_provinsi", "provinsi"],
        how="left"
    )

    print("\nFact Table :", fact_table.shape)

    return fact_table


# ==================================================
# FEATURE ENGINEERING
# ==================================================

def create_features(fact_table):

    fact_table["konsumsi_per_1000_penduduk"] = (
        fact_table["konsumsi_pangan"]
        / fact_table["jumlah_penduduk"]
    ) * 1000

    fact_table["persentase_undernourish"] = (
        fact_table["penduduk_undernourish"]
        / fact_table["jumlah_penduduk"]
    ) * 100

    fact_table["log_penduduk"] = np.log1p(
        fact_table["jumlah_penduduk"]
    )

    numeric_cols = [
        "konsumsi_pangan",
        "pou",
        "jumlah_penduduk",
        "penduduk_undernourish"
    ]

    for col in numeric_cols:
        if col in fact_table.columns:
            fact_table[col] = pd.to_numeric(
                fact_table[col],
                errors="coerce"
            )

    return fact_table


# ==================================================
# DATASET PREPARATION
# ==================================================

def build_dashboard_dataset(fact_table):

    dashboard_dataset = fact_table.copy()

    dashboard_dataset.to_csv(
        "dashboard_dataset.csv",
        index=False
    )

    return dashboard_dataset


def build_ml_dataset(fact_table):

    ml_dataset = fact_table.copy()

    ml_dataset["provinsi_id"] = (
        ml_dataset["provinsi"]
        .astype("category")
        .cat.codes
    )

    ml_dataset["kelompok_id"] = (
        ml_dataset["kelompok_bahan_pangan"]
        .astype("category")
        .cat.codes
    )

    ml_dataset["komoditas_id"] = (
        ml_dataset["komoditas"]
        .astype("category")
        .cat.codes
    )

    ml_dataset = ml_dataset[
        [
            "tahun",
            "provinsi",
            "provinsi_id",
            "kelompok_bahan_pangan",
            "kelompok_id",
            "komoditas",
            "komoditas_id",
            "konsumsi_pangan",
            "jumlah_penduduk",
            "penduduk_undernourish",
            "persentase_undernourish",
            "log_penduduk",
            "pou"
        ]
    ]

    ml_dataset = ml_dataset.dropna()

    ml_dataset.to_csv(
        "ml_dataset.csv",
        index=False
    )

    return ml_dataset


# ==================================================
# REPORTING
# ==================================================

def generate_report(
    fact_table,
    dashboard_dataset,
    ml_dataset,
    df_nutrition
):

    print("\n=== DATA QUALITY ===")

    print(
        "Dashboard Dataset :",
        dashboard_dataset.shape
    )

    print(
        "ML Dataset :",
        ml_dataset.shape
    )

    print(
        "Total Provinsi :",
        fact_table["provinsi"].nunique()
    )

    print(
        "Total Komoditas :",
        fact_table["komoditas"].nunique()
    )

    print(
        "Rentang Tahun :",
        fact_table["tahun"].min(),
        "-",
        fact_table["tahun"].max()
    )

    print("\n=== SAMPLE DATA ===")
    print(dashboard_dataset.head())

    print("\n=== SAMPLE DATA NUTRITION ===")
    print(df_nutrition.head())

    print("\nFile Generated:")
    print("dashboard_dataset.csv")
    print("ml_dataset.csv")


# ==================================================
# DATABASE
# ==================================================

def create_db_engine():

    return create_engine(
        f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
    )


def create_tables(engine):

    with engine.begin() as conn:

        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS dashboard_food_security (
                id BIGINT AUTO_INCREMENT PRIMARY KEY,
                tahun INT,
                kode_provinsi INT,
                provinsi VARCHAR(100),
                kelompok_bahan_pangan VARCHAR(100),
                komoditas VARCHAR(100),
                konsumsi_pangan DOUBLE,
                pou DOUBLE,
                jumlah_penduduk BIGINT,
                penduduk_undernourish BIGINT,
                konsumsi_per_1000_penduduk DOUBLE,
                persentase_undernourish DOUBLE,
                log_penduduk DOUBLE
            )
        """))

        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS master_nutrition (
                id BIGINT PRIMARY KEY,
                name VARCHAR(255),
                calories DOUBLE,
                proteins DOUBLE,
                fat DOUBLE,
                carbohydrate DOUBLE,
                image TEXT,
                INDEX idx_name (name)
            )
        """))


def load_to_database(
    engine,
    dashboard_dataset,
    df_nutrition
):

    dashboard_dataset.to_sql(
        "dashboard_food_security",
        engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi"
    )

    df_nutrition.to_sql(
        "master_nutrition",
        engine,
        if_exists="append",
        index=False,
        chunksize=1000,
        method="multi"
    )


# ==================================================
# PIPELINE
# ==================================================

def run_pipeline():

    # EXTRACT
    df_pou, df_food, df_nutrition = extract_data()

    # TRANSFORM
    df_pou, df_food, df_nutrition = transform_data(
        df_pou,
        df_food,
        df_nutrition
    )

    # FACT TABLE
    fact_table = build_fact_table(
        df_food,
        df_pou
    )

    # FEATURE ENGINEERING
    fact_table = create_features(
        fact_table
    )

    # DATASET
    dashboard_dataset = build_dashboard_dataset(
        fact_table
    )

    ml_dataset = build_ml_dataset(
        fact_table
    )

    # REPORT
    generate_report(
        fact_table,
        dashboard_dataset,
        ml_dataset,
        df_nutrition
    )

    # LOAD
    engine = create_db_engine()

    create_tables(engine)

    load_to_database(
        engine,
        dashboard_dataset,
        df_nutrition
    )

    print("\n=== PIPELINE SUCCESS ===")


if __name__ == "__main__":
    run_pipeline()

=== DATA LOADED ===
POU       : (242, 7)
FOOD      : (8227, 8)
NUTRITION : (1346, 7)

Fact Table : (8227, 9)

=== DATA QUALITY ===
Dashboard Dataset : (8227, 12)
ML Dataset : (3817, 13)
Total Provinsi : 39
Total Komoditas : 36
Rentang Tahun : 2018 - 2024

=== SAMPLE DATA ===
   tahun  kode_provinsi provinsi kelompok_bahan_pangan    komoditas  \
0   2018             11     Aceh           Padi-Padian  Padi-padian   
1   2018             11     Aceh           Padi-Padian        Beras   
2   2018             11     Aceh           Padi-Padian       Jagung   
3   2018             11     Aceh           Padi-Padian       Terigu   
4   2018             11     Aceh           Umbi-umbian  Umbi-umbian   

   konsumsi_pangan  pou  jumlah_penduduk  penduduk_undernourish  \
0            118.5  NaN              NaN                    NaN   
1            103.1  NaN              NaN                    NaN   
2              0.4  NaN              NaN                    NaN   
3             15.0  NaN      